In [1]:
import os
import time
import json
import math
import torch
import librosa
import numpy as np
import itertools
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.tensorboard import SummaryWriter
import warnings

ROOT_DIR = Path(os.getcwd()).parent
MODELS_DIR = ROOT_DIR / "models"
LOGS_DIR = ROOT_DIR / "logs"

for d in [MODELS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def lengths_to_padding_mask(lengths, max_len=None):
    if max_len is None:
        max_len = int(lengths.max().item())
    steps = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return steps >= lengths.unsqueeze(1)  # True means padding


class SinusoidalPositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        self.d_model = d_model
        self.register_buffer("pe", torch.empty(1, 0, d_model), persistent=False)
        self._extend_pe(max_len)

    def _extend_pe(self, max_len):
        if max_len <= self.pe.size(1):
            return
        pe = torch.zeros(max_len, self.d_model, device=self.pe.device)
        position = torch.arange(0, max_len, dtype=torch.float32, device=self.pe.device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.d_model, 2, dtype=torch.float32, device=self.pe.device) * (-math.log(10000.0) / self.d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        self._extend_pe(x.size(1))
        return x + self.pe[:, : x.size(1)].to(dtype=x.dtype, device=x.device)


class AudioToBlendshapeModel(torch.nn.Module):
    def __init__(
        self,
        num_blendshapes=52,
        hidden_dim=384,
        num_layers=3,
        num_heads=6,
        dropout=0.1,
        max_tokens=1200,
    ):
        super().__init__()
        self.max_tokens = int(max_tokens)

        # Strided conv frontend gives a strong quality/speed tradeoff.
        self.frontend_specs = [(7, 2, 3), (5, 2, 2), (5, 2, 2)]
        self.audio_backbone = torch.nn.Sequential(
            torch.nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3, bias=False),
            torch.nn.BatchNorm1d(64),
            torch.nn.SiLU(),
            torch.nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2, bias=False),
            torch.nn.BatchNorm1d(128),
            torch.nn.SiLU(),
            torch.nn.Conv1d(128, hidden_dim, kernel_size=5, stride=2, padding=2, bias=False),
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.SiLU(),
        )
        self.temporal_mixer = torch.nn.Sequential(
            torch.nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2, groups=hidden_dim, bias=False),
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.SiLU(),
            torch.nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1),
        )

        self.input_norm = torch.nn.LayerNorm(hidden_dim)
        self.pos_enc = SinusoidalPositionalEncoding(hidden_dim)

        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.transformer = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head_norm = torch.nn.LayerNorm(hidden_dim)

        self.regressor = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim // 2),
            torch.nn.SiLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_dim // 2, num_blendshapes),
            torch.nn.Sigmoid(),
        )

    def _downsample_lengths(self, lengths):
        out = lengths
        for k, s, p in self.frontend_specs:
            out = torch.div(out + 2 * p - (k - 1) - 1, s, rounding_mode="floor") + 1
            out = torch.clamp(out, min=1)
        return out

    def forward(self, x, lengths=None):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        y = self.audio_backbone(x)
        y = y + self.temporal_mixer(y)
        x = y.transpose(1, 2)

        if lengths is None:
            out_lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)
        else:
            out_lengths = self._downsample_lengths(lengths)

        # Keep transformer attention cost bounded for long clips.
        if x.size(1) > self.max_tokens:
            pool_stride = int(math.ceil(x.size(1) / self.max_tokens))
            x = x.transpose(1, 2)
            x = F.avg_pool1d(x, kernel_size=pool_stride, stride=pool_stride, ceil_mode=True)
            x = x.transpose(1, 2)
            out_lengths = torch.div(out_lengths + pool_stride - 1, pool_stride, rounding_mode="floor")
            out_lengths = torch.clamp(out_lengths, min=1)

        x = self.input_norm(x)
        x = self.pos_enc(x)
        padding_mask = lengths_to_padding_mask(out_lengths, x.size(1))
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        x = self.head_norm(x)
        return self.regressor(x), out_lengths

In [6]:
class BEATDataset(Dataset):
    def __init__(self, beat_root, target_sr=16000, max_samples=None):
        self.beat_root = Path(beat_root)
        self.target_sr = target_sr
        self.items = []

        wav_candidates = sorted(self.beat_root.rglob("*.wav"))
        for wav_path in wav_candidates:
            json_path = wav_path.with_suffix(".json")
            if json_path.exists():
                self.items.append((wav_path, json_path))

        if max_samples is not None:
            self.items = self.items[:max_samples]

        if len(self.items) == 0:
            raise FileNotFoundError(
                f"No BEAT wav+json pairs found under {self.beat_root}. "
                "Ensure files like <sample>.wav and <sample>.json exist side-by-side."
            )

    def __len__(self):
        return len(self.items)

    def _load_target(self, target_path):
        with open(target_path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        frames = []
        if isinstance(obj, dict):
            if isinstance(obj.get("frames"), list):
                frames = obj["frames"]
            elif isinstance(obj.get("weights"), list):
                frames = [{"weights": obj["weights"]}]
        elif isinstance(obj, list):
            frames = obj

        weights = []
        for fr in frames:
            if isinstance(fr, dict) and isinstance(fr.get("weights"), list):
                weights.append(fr["weights"])
            elif isinstance(fr, list):
                weights.append(fr)

        if len(weights) == 0:
            raise ValueError(f"No frame weights in {target_path}")

        tgt = torch.tensor(weights, dtype=torch.float32)

        if tgt.dim() == 1:
            tgt = tgt.unsqueeze(0)
        elif tgt.dim() > 2:
            tgt = tgt.reshape(tgt.shape[0], -1)

        feat = tgt.shape[1]
        if feat < 52:
            pad = torch.zeros(tgt.shape[0], 52 - feat, dtype=tgt.dtype)
            tgt = torch.cat([tgt, pad], dim=1)
        elif feat > 52:
            tgt = tgt[:, :52]

        return tgt

    def __getitem__(self, idx):
        audio_path, target_path = self.items[idx]
        audio_np, _ = librosa.load(audio_path, sr=self.target_sr, mono=True, duration=10.0)
        target = self._load_target(target_path)

        fps = 60.0 
        max_frames = int((len(audio_np) / self.target_sr) * fps)
        target = target[:max_frames]

        audio = torch.tensor(audio_np, dtype=torch.float32)
        return audio, target


def collate_fn(batch):
    audio_seqs, target_seqs = zip(*batch)
    audio_lengths = torch.tensor([x.shape[0] for x in audio_seqs], dtype=torch.long)
    target_lengths = torch.tensor([x.shape[0] for x in target_seqs], dtype=torch.long)

    audio_padded = pad_sequence(audio_seqs, batch_first=True, padding_value=0.0)
    target_padded = pad_sequence(target_seqs, batch_first=True, padding_value=0.0)
    return audio_padded, target_padded, audio_lengths, target_lengths

In [15]:
def length_mask(lengths, max_len, device):
    steps = torch.arange(max_len, device=device).unsqueeze(0)
    return steps < lengths.unsqueeze(1)


def masked_mse(pred, target, lengths):
    mask = length_mask(lengths, target.size(1), target.device).unsqueeze(-1).float()
    diff2 = (pred - target).pow(2) * mask
    denom = (mask.sum() * pred.size(-1)).clamp_min(1.0)
    return diff2.sum() / denom


def masked_mae(pred, target, lengths):
    mask = length_mask(lengths, target.size(1), target.device).unsqueeze(-1).float()
    diff = (pred - target).abs() * mask
    denom = (mask.sum() * pred.size(-1)).clamp_min(1.0)
    return diff.sum() / denom


def masked_accuracy(pred, target, lengths, threshold=0.05):
    mask = length_mask(lengths, target.size(1), target.device).unsqueeze(-1).float()
    abs_err = (pred - target).abs()
    correct = (abs_err <= threshold).float() * mask
    denom = (mask.sum() * pred.size(-1)).clamp_min(1.0)
    return correct.sum() / denom


def _fmt_seconds(total_seconds):
    total_seconds = max(0, int(total_seconds))
    hours, rem = divmod(total_seconds, 3600)
    minutes, seconds = divmod(rem, 60)
    if hours > 0:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes > 0:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    scaler,
    vel_weight=0.35,
    acc_threshold=0.05,
    progress_prefix="",
    progress_every=20,
    ):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0
    epoch_start = time.time()
    total_batches = max(1, len(loader))

    for batch_idx, (audio, target, audio_lengths, target_lengths) in enumerate(loader, start=1):
        if audio.numel() == 0 or target.numel() == 0:
            continue

        audio = audio.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        audio_lengths = audio_lengths.to(device, non_blocking=True)
        target_lengths = target_lengths.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            output, _ = model(audio, audio_lengths)  # [B, T_pred, 52]

            if output.shape[1] != target.shape[1]:
                output = output.transpose(1, 2)
                output = F.interpolate(output, size=target.shape[1], mode="linear", align_corners=False)
                output = output.transpose(1, 2)

            mse = masked_mse(output, target, target_lengths)

            if output.size(1) > 1:
                pred_vel = output[:, 1:] - output[:, :-1]
                tgt_vel = target[:, 1:] - target[:, :-1]
                vel_lengths = torch.clamp(target_lengths - 1, min=0)
                vel_loss = masked_mse(pred_vel, tgt_vel, vel_lengths)
            else:
                vel_loss = torch.zeros((), device=device)

            loss = mse + vel_weight * vel_loss

        if not torch.isfinite(loss):
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        with torch.no_grad():
            acc = masked_accuracy(output, target, target_lengths, threshold=acc_threshold)

        total_loss += loss.item()
        total_acc += acc.item()
        num_batches += 1

        if batch_idx == 1 or batch_idx % progress_every == 0 or batch_idx == total_batches:
            elapsed = time.time() - epoch_start
            avg_batch_seconds = elapsed / max(1, batch_idx)
            eta_seconds = avg_batch_seconds * max(0, total_batches - batch_idx)
            print(
                f"{progress_prefix}Batch {batch_idx}/{total_batches} | "
                f"Avg Loss: {total_loss / max(1, num_batches):.6f}, "
                f"Avg Acc: {total_acc / max(1, num_batches):.4f} | "
                f"Elapsed: {_fmt_seconds(elapsed)} | ETA: {_fmt_seconds(eta_seconds)}",
                flush=True,
            )

    denom = max(1, num_batches)
    return total_loss / denom, total_acc / denom


def validate(
    model,
    loader,
    device,
    acc_threshold=0.05,
    progress_prefix="",
    progress_every=20,
    ):
    model.eval()
    val_loss = 0.0
    total_mae = 0.0
    total_acc = 0.0
    num_batches = 0
    val_start = time.time()
    total_batches = max(1, len(loader))

    with torch.no_grad():
        for batch_idx, (audio, target, audio_lengths, target_lengths) in enumerate(loader, start=1):
            if audio.numel() == 0 or target.numel() == 0:
                continue

            audio = audio.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            audio_lengths = audio_lengths.to(device, non_blocking=True)
            target_lengths = target_lengths.to(device, non_blocking=True)

            output, _ = model(audio, audio_lengths)

            if output.shape[1] != target.shape[1]:
                output = output.transpose(1, 2)
                output = F.interpolate(output, size=target.shape[1], mode="linear", align_corners=False)
                output = output.transpose(1, 2)

            loss = masked_mse(output, target, target_lengths)
            mae = masked_mae(output, target, target_lengths)
            acc = masked_accuracy(output, target, target_lengths, threshold=acc_threshold)

            if not (torch.isfinite(loss) and torch.isfinite(mae) and torch.isfinite(acc)):
                continue

            val_loss += loss.item()
            total_mae += mae.item()
            total_acc += acc.item()
            num_batches += 1

            if batch_idx == 1 or batch_idx % progress_every == 0 or batch_idx == total_batches:
                elapsed = time.time() - val_start
                avg_batch_seconds = elapsed / max(1, batch_idx)
                eta_seconds = avg_batch_seconds * max(0, total_batches - batch_idx)
                print(
                    f"{progress_prefix}Val Batch {batch_idx}/{total_batches} | "
                    f"Avg Val Loss: {val_loss / max(1, num_batches):.6f}, "
                    f"Avg Val Acc: {total_acc / max(1, num_batches):.4f} | "
                    f"Elapsed: {_fmt_seconds(elapsed)} | ETA: {_fmt_seconds(eta_seconds)}",
                    flush=True,
                )

    denom = max(1, num_batches)
    return val_loss / denom, total_mae / denom, total_acc / denom


def run_trial(config, train_loader, val_loader, trial_set_dir=None, trial_index=None, total_trials=None):
    if trial_set_dir is None:
        trial_set_dir = LOGS_DIR
    trial_set_dir = Path(trial_set_dir)
    trial_set_dir.mkdir(parents=True, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    lr_str = f"{config['lr']:.0e}".replace("e-0", "e-")
    dropout_str = f"{config['dropout']:.2f}".rstrip("0").rstrip(".")
    
    # This string is now guaranteed unique by trial_index
    trial_id = (
        f"Trial_{trial_index:03d}_{timestamp}"
        f"_lr{lr_str}_h{config['hidden_dim']}_L{config['num_layers']}"
        f"_H{config['num_heads']}_d{dropout_str}"
    )
    
    run_id = f"RUN_{trial_id}"
    # Use the unique trial_id for the SummaryWriter path
    writer = SummaryWriter(log_dir=trial_set_dir / trial_id)

    model = AudioToBlendshapeModel(
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        dropout=config["dropout"],
        max_tokens=int(config.get("max_tokens", 1200)),
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config.get("weight_decay", 1e-2),
        betas=(0.9, 0.98),
    )

    total_steps = max(1, config["epochs"] * len(train_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config["lr"],
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=100.0,
    )
    scaler_enabled = (device.type == "cuda")
    if hasattr(torch.amp, "GradScaler"):
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=scaler_enabled)
        except Exception:
            scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)

    best_val_loss = float("inf")
    best_epoch = 0
    best_val_acc = 0.0
    patience = int(config.get("early_stopping_patience", 8))
    min_delta = float(config.get("early_stopping_min_delta", 1e-4))
    epochs_no_improve = 0
    acc_threshold = float(config.get("acc_threshold", 0.05))

    trial_start = time.time()
    epoch_durations = []
    val_acc_history = []
    total_epochs = int(config["epochs"])
    if trial_index is not None and total_trials is not None:
        trial_prefix = f"[Trial {trial_index}/{total_trials}] "
    else:
        trial_prefix = ""
    print(
        f"{trial_prefix}Batches -> train: {len(train_loader)}, val: {len(val_loader)}",
        flush=True,
    )

    for epoch in range(total_epochs):
        epoch_start = time.time()
        print(f"{trial_prefix}Epoch {epoch+1}/{total_epochs} starting...", flush=True)

        t_loss, t_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            scaler,
            vel_weight=config.get("vel_weight", 0.35),
            acc_threshold=acc_threshold,
            progress_prefix=f"{trial_prefix}Epoch {epoch+1}/{total_epochs} | ",
            progress_every=max(1, len(train_loader) // 10),
        )
        v_loss, v_mae, v_acc = validate(
            model,
            val_loader,
            device,
            acc_threshold=acc_threshold,
            progress_prefix=f"{trial_prefix}Epoch {epoch+1}/{total_epochs} | ",
            progress_every=max(1, len(val_loader) // 5),
        )
        val_acc_history.append(v_acc)

        epoch_seconds = time.time() - epoch_start
        epoch_durations.append(epoch_seconds)
        avg_epoch_seconds = sum(epoch_durations) / len(epoch_durations)
        epochs_left = total_epochs - (epoch + 1)
        trial_eta_seconds = avg_epoch_seconds * epochs_left
        trial_elapsed_seconds = time.time() - trial_start

        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"{trial_prefix}Epoch {epoch+1}/{total_epochs} complete -> "
            f"Train Loss: {t_loss:.6f}, Val Loss: {v_loss:.6f}, "
            f"Train Acc: {t_acc:.4f}, Val Acc: {v_acc:.4f}, Val MAE: {v_mae:.6f}, "
            f"LR: {current_lr:.2e} | "
            f"Epoch Time: {_fmt_seconds(epoch_seconds)} | "
            f"Trial Elapsed: {_fmt_seconds(trial_elapsed_seconds)} | "
            f"Trial ETA: {_fmt_seconds(trial_eta_seconds)}",
            flush=True,
        )

        # High-signal TensorBoard metrics only.
        writer.add_scalar("Loss/Train", t_loss, epoch)
        writer.add_scalar("Loss/Val", v_loss, epoch)
        writer.add_scalar("Accuracy/Train", t_acc, epoch)
        writer.add_scalar("Accuracy/Val", v_acc, epoch)
        writer.add_scalar("Error/Val_MAE", v_mae, epoch)
        writer.add_scalar("Optimization/LR", current_lr, epoch)

        if v_loss < (best_val_loss - min_delta):
            best_val_loss = v_loss
            best_val_acc = v_acc
            best_epoch = epoch + 1
            epochs_no_improve = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": config,
                    "val_loss": best_val_loss,
                    "val_accuracy": best_val_acc,
                },
                MODELS_DIR / f"{run_id}_best.pt",
            )
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(
                f"{trial_prefix}Early stopping at epoch {epoch+1}. "
                f"Best epoch: {best_epoch}, best val loss: {best_val_loss:.6f}, best val acc: {best_val_acc:.4f}",
                flush=True,
            )
            break

    trial_total_seconds = time.time() - trial_start
    trial_acc = float(sum(val_acc_history) / max(1, len(val_acc_history)))
    print(
        f"{trial_prefix}Trial finished in {_fmt_seconds(trial_total_seconds)}. "
        f"Best epoch: {best_epoch}, best val loss: {best_val_loss:.6f}, "
        f"best val acc: {best_val_acc:.4f}, trial acc: {trial_acc:.4f}",
        flush=True,
    )

    writer.add_scalar("Accuracy/Trial", trial_acc, 0)
    writer.add_scalar("Summary/Best_Val_Loss", best_val_loss, 0)
    writer.add_scalar("Summary/Best_Val_Accuracy", best_val_acc, 0)
    writer.add_scalar("Summary/Best_Epoch", float(best_epoch), 0)

    writer.add_hparams(
        config,
        {
            "hparam/best_val_loss": best_val_loss,
            "hparam/best_val_accuracy": best_val_acc,
            "hparam/trial_accuracy": trial_acc,
            "hparam/best_epoch": float(best_epoch),
        },
    )
    writer.close()
    return best_val_loss

In [18]:
# Local BEAT dataset root
BEAT_ROOT = ROOT_DIR / "data" / "beat_english_v0.2.1" / "beat_english_v0.2.1"

# Set to an integer like 240 for quick experiments
MAX_SAMPLES = None

full_dataset = BEATDataset(BEAT_ROOT, target_sr=16000, max_samples=MAX_SAMPLES)
print(f"Loaded BEAT wav+json pairs: {len(full_dataset)}")

NUM_WORKERS = min(4, os.cpu_count() or 1)
NUM_WORKERS = 0  # Set to 0 temporarily to see if it's a multiprocessing issue
if os.name == "nt":
    NUM_WORKERS = min(2, NUM_WORKERS)
PIN_MEMORY = (device.type == "cuda")
PERSISTENT_WORKERS = NUM_WORKERS > 0
PERSISTENT_WORKERS = False
print(f"DataLoader workers: {NUM_WORKERS} | pin_memory: {PIN_MEMORY}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
split_gen = torch.Generator().manual_seed(42)
train_set, val_set = random_split(full_dataset, [train_size, val_size], generator=split_gen)


def make_loaders(batch_size=8, shuffle_train=True):
    kwargs = dict(
        batch_size=int(batch_size),
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    train_loader = DataLoader(train_set, shuffle=shuffle_train, **kwargs)
    val_loader = DataLoader(val_set, shuffle=False, **kwargs)
    return train_loader, val_loader


# Quick batch sanity check
sanity_loader, _ = make_loaders(batch_size=4, shuffle_train=True)
audio_b, target_b, audio_len_b, target_len_b = next(iter(sanity_loader))
print(f"Batch Audio: {audio_b.shape}, lengths: min={audio_len_b.min().item()}, max={audio_len_b.max().item()}")
print(f"Batch Targets: {target_b.shape}, lengths: min={target_len_b.min().item()}, max={target_len_b.max().item()}")
del sanity_loader

Loaded BEAT wav+json pairs: 1945
DataLoader workers: 0 | pin_memory: True
Batch Audio: torch.Size([4, 160000]), lengths: min=160000, max=160000
Batch Targets: torch.Size([4, 600, 52]), lengths: min=600, max=600


In [ ]:
# search_space = {
#     "lr": [3e-4, 1e-4],
#     "hidden_dim": [384, 512],
#     "num_layers": [3, 4],
#     "num_heads": [6, 8],
#     "dropout": [0.0, 0.1],
#     "weight_decay": [1e-2],
#     "vel_weight": [0.35],
#     "acc_threshold": [0.05],
#     "early_stopping_patience": [8],
#     "early_stopping_min_delta": [1e-4],
#     "batch_size": [8],
#     "epochs": [5],
# }

search_space = {
    "lr": [1e-4, 8e-5],
    "hidden_dim": [512],
    "num_layers": [4],
    "num_heads": [8],
    "dropout": [0.1],
    "weight_decay": [1e-2],
    "vel_weight": [0.35],
    "acc_threshold": [0.05],
    "early_stopping_patience": [8],
    "early_stopping_min_delta": [1e-4],
    "batch_size": [8],
    "epochs": [50],
}

keys, values = zip(*search_space.items())
trials = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total Trials to Run: {len(trials)}")

def fmt_duration(seconds):
    seconds = max(0, int(seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours > 0:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    if minutes > 0:
        return f"{minutes}m {secs:02d}s"
    return f"{secs}s"


trial_set_timestamp = time.strftime("%Y%m%d_%H%M%S")
trial_set_dir = LOGS_DIR / f"SWEEP_{trial_set_timestamp}"
trial_set_dir.mkdir(parents=True, exist_ok=True)
print(f"Trial set log dir: {trial_set_dir}")

sweep_start = time.time()
trial_durations = []
total_trials = len(trials)

for i, cfg in enumerate(trials):
    completed_trials = i
    sweep_elapsed = time.time() - sweep_start

    if trial_durations:
        avg_trial_seconds = sum(trial_durations) / len(trial_durations)
        remaining_trials = total_trials - completed_trials
        sweep_eta = avg_trial_seconds * remaining_trials
        eta_text = fmt_duration(sweep_eta)
    else:
        eta_text = "estimating..."

    print(
        f"\nSweep Progress: {completed_trials}/{total_trials} complete | "
        f"Elapsed: {fmt_duration(sweep_elapsed)} | "
        f"ETA: {eta_text}"
    )
    print(f"--- [Trial {i+1}/{total_trials}] Config: {cfg} ---")

    t_loader = DataLoader(
        train_set, 
        batch_size=cfg["batch_size"], 
        shuffle=True, 
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=False  # Changed for sweep stability
    )
    v_loader = DataLoader(
        val_set, 
        batch_size=cfg["batch_size"], 
        shuffle=False, 
        collate_fn=collate_fn,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=False
    )

    trial_start = time.time()
    try:
        res = run_trial(
            cfg,
            t_loader,
            v_loader,
            trial_set_dir=trial_set_dir,
            trial_index=i + 1,
            total_trials=total_trials,
        )
        print(f"Success. Best Val Loss: {res:.6f}")
    except Exception as e:
        print(f"Trial {i+1} crashed: {e}")

    trial_elapsed = time.time() - trial_start
    trial_durations.append(trial_elapsed)
    print(f"Trial Duration: {fmt_duration(trial_elapsed)}")

sweep_total = time.time() - sweep_start
print(f"\nSweep complete in {fmt_duration(sweep_total)}")

Total Trials to Run: 2
Trial set log dir: D:\Coding\project-group-10\logs\SWEEP_20260407_220432

Sweep Progress: 0/2 complete | Elapsed: 0s | ETA: estimating...
--- [Trial 1/2] Config: {'lr': 0.0001, 'hidden_dim': 512, 'num_layers': 4, 'num_heads': 8, 'dropout': 0.1, 'weight_decay': 0.01, 'vel_weight': 0.35, 'acc_threshold': 0.05, 'early_stopping_patience': 8, 'early_stopping_min_delta': 0.0001, 'batch_size': 8, 'epochs': 50} ---
[Trial 1/2] Batches -> train: 195, val: 49
[Trial 1/2] Epoch 1/50 starting...
[Trial 1/2] Epoch 1/50 | Batch 1/195 | Avg Loss: 0.169292, Avg Acc: 0.0120 | Elapsed: 2s | ETA: 9m 26s
[Trial 1/2] Epoch 1/50 | Batch 19/195 | Avg Loss: 0.142761, Avg Acc: 0.0232 | Elapsed: 32s | ETA: 5m 04s
[Trial 1/2] Epoch 1/50 | Batch 38/195 | Avg Loss: 0.127219, Avg Acc: 0.0266 | Elapsed: 57s | ETA: 3m 56s
[Trial 1/2] Epoch 1/50 | Batch 57/195 | Avg Loss: 0.115956, Avg Acc: 0.0305 | Elapsed: 1m 20s | ETA: 3m 15s
[Trial 1/2] Epoch 1/50 | Batch 76/195 | Avg Loss: 0.107116, Avg Acc